# Case 1 — Cylinder Flow: POD-AS-PRS Workflow

This notebook reproduces the drag-coefficient ($C_d$) surrogate study for a
2-D cylinder at $Re = 100$ using the full POD-AS-PRS pipeline:

1. **Preprocessing** – merge Nek5000 snapshots and compute vorticity
2. **POD** – decompose vorticity snapshots via SVD
3. **ResNet** – train a fully-connected ResNet to map POD coefficients → $C_d$
4. **Gradient analysis** – compute autograd gradients and validate against FD
5. **Active Subspaces (AS)** – identify the dominant input directions
6. **Polynomial Response Surface (PRS)** – fit and evaluate the low-dimensional surrogate


## 0 · Setup

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import torch

from core.pod_engine       import POD_SVD
from core.resnet_model     import ResNet
from core.resnet_trainer   import (set_random_seed, load_or_train,
                                    evaluate_and_save_metrics,
                                    plot_loss_curve, plot_prediction_comparison,
                                    compute_all_gradients)
from core.gradient_analysis import compare_gradients_nature_style_dataset
from utils.data_loader      import (load_and_preprocess_data, denormalise,
                                     load_pod_vis_data)
from utils.visualization    import (plot_pod_importance, plot_response_surface_2d,
                                     validate_response_surface,
                                     compare_rom_fom_predictions,
                                     plot_polynomial_cv,
                                     plot_subspace_polynomial_heatmap,
                                     plot_interaction_heatmap,
                                     plot_pod_energy,
                                     plot_eigenvalues,
                                     plot_pod_modes_and_coeffs,
                                     plot_pod_phase_space_triangle,
                                     plot_mesh_and_vorticity,
                                     plot_qoi)
import lib.active_subspaces as ac

set_random_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

Random seed set to: 42
Device: cuda


## 1 · Data Paths
Adjust these paths to match your local directory layout.

In [2]:
FLOW_DATA_PATH = '../data/Case1_Cylinder/flow_field_data.npz'
QOI_DATA_PATH  = '../data/Case1_Cylinder/drag_coefficient.dat'
RESULTS_DIR    = '../results/Case1_Cylinder'
MODEL_PATH     = os.path.join(RESULTS_DIR, 'resnet_model.pth')

NUM_POD_COEFFS = 150
os.makedirs(RESULTS_DIR, exist_ok=True)

## 1.5 · Computational Mesh and Vorticity Field

In [3]:
NEK_FILE = '/mnt/data/bak/HD5/NekExamples/ext_cyl/ext_cyl0.f00001'

plot_mesh_and_vorticity(
    flow_data_path=FLOW_DATA_PATH,
    geometry='cylinder',
    nek_data_path=NEK_FILE,
    snapshot_idx=10,
    save_dir=os.path.join(RESULTS_DIR, 'Mesh'),
)

Mesh/vorticity plot saved to ../results/Case1_Cylinder/Mesh/mesh.[jpg|pdf]


## 2 · Load Data and Run POD

In [ ]:
(
    train_loader, val_loader, test_loader,
    pod_coeffs, pod_coeffs_norm,
    pod_mean, pod_std,
    qoi_mean, qoi_std,
    pod_min, pod_max,
    qoi_min, qoi_max,
) = load_and_preprocess_data(
    flow_data_path=FLOW_DATA_PATH,
    qoi_data_path=QOI_DATA_PATH,
    num_pod_coeffs=NUM_POD_COEFFS,
    train_ratio=0.8,
    val_ratio=0.1,
    batch_size=32,
    apply_region_filter=False,          # No spatial mask for cylinder
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

print(f'POD coefficients shape : {pod_coeffs.shape}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

# POD visualisation — energy spectrum, spatial modes + time coefficients, phase portrait
pod_vis = load_pod_vis_data(
    pod_save_dir=os.path.join(RESULTS_DIR, 'POD'),
    flow_data_path=FLOW_DATA_PATH,
    apply_region_filter=False,
)

plot_pod_energy(
    pod_vis['Ds'], num_modes=20,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='cylinder',
)

plot_eigenvalues(
    pod_vis['S'], num_values=20,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='cylinder',
)

plot_pod_modes_and_coeffs(
    pod_vis['PhiU'], pod_vis['An'],
    pod_vis['original_shape'],
    pod_vis['x_grid'], pod_vis['y_grid'],
    num_modes=6, geometry='cylinder',
    region_mask=pod_vis['region_mask'],
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
)

plot_pod_phase_space_triangle(
    pod_vis['An'], num_modes=6,
    save_dir=os.path.join(RESULTS_DIR, 'POD'),
    geometry='cylinder',
)


Loading flow field data...
Flow data keys: ['coords', 'times', 'velocity', 'pressure', 'vorticity', 'vorticity_grid_x', 'vorticity_grid_y', 'n_timesteps', 'n_elements', 'n_components', 'n_points', 'start_time', 'end_time', 'start_idx', 'end_idx', 'mesh_limits_x', 'mesh_limits_y', 'vorticity_nx', 'vorticity_ny']
Vorticity array shape: (1000, 356, 593)
Grid range: X=[-15.00, 35.00], Y=[-15.00, 15.00]
Reshaped vorticity: (1000, 211108)
Loading cached POD data from: ../results/Case1_Cylinder/POD/pod_data.npz

Loading QoI data from: ../data/Case1_Cylinder/drag_coefficient.dat
QoI shape: (1000, 1), range: [1.350051, 1.369004]
POD coefficients used: (1000, 150)
QoI values used:       (1000, 1)
POD coefficients shape : (1000, 150)
Train batches: 25, Val: 4, Test: 4
POD energy plot saved to ../results/Case1_Cylinder/POD/pod_energy.[jpg|pdf]
POD modes & coefficients plot saved to ../results/Case1_Cylinder/POD/pod_modes_and_coeffs.[jpg|pdf]
POD phase-space triangle plot saved to ../results/Case1

## 3 · Build and Train the ResNet Surrogate

In [5]:
set_random_seed(42)   # reset right before model creation — matches legacy train.py line 88

# Match the exact legacy architecture: hidden_size=128, num_blocks=7
NUM_BLOCKS = 7
model = ResNet(input_size=NUM_POD_COEFFS, hidden_size=128,
               num_blocks=NUM_BLOCKS, dropout_rate=0.1).to(DEVICE)
print(model)

model, train_losses, val_losses = load_or_train(
    model, train_loader, val_loader, DEVICE,
    model_save_path=MODEL_PATH,
    num_epochs=1000,
    patience=100,
    lr=0.001,
)

plot_loss_curve(train_losses, val_losses, patience=100,
                results_dir=RESULTS_DIR)

Random seed set to: 42
ResNet(
  (input_layer): Sequential(
    (0): Linear(in_features=150, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (res_blocks): ModuleList(
    (0-6): 7 x ResidualBlock(
      (block): Sequential(
        (0): Linear(in_features=128, out_features=128, bias=True)
        (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Dropout(p=0.1, inplace=False)
        (4): Linear(in_features=128, out_features=128, bias=True)
        (5): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (relu): ReLU()
    )
  )
  (output_layer): Sequential(
    (0): Dropout(p=0.1, inplace=False)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.1, in

## 4 · Evaluate the Surrogate

In [6]:
denorm = lambda v: denormalise(v, qoi_mean, qoi_std)

metrics = evaluate_and_save_metrics(
    model,
    loaders=[train_loader, val_loader, test_loader],
    split_names=['Train', 'Validation', 'Test'],
    device=DEVICE,
    denorm_fn=denorm,
    results_dir=RESULTS_DIR,
)

Train: MSE=3.176605e-08, MAE=1.435891e-04, R²=0.999292, MaxRelErr=0.000379
Validation: MSE=1.105010e-07, MAE=2.752221e-04, R²=0.997608, MaxRelErr=0.000587
Test: MSE=9.992280e-08, MAE=2.477801e-04, R²=0.997817, MaxRelErr=0.000730
Metrics saved to ../results/Case1_Cylinder/metrics.txt


## 5 · Gradient Analysis (Autograd vs Finite Difference)

In [7]:
# Collect only training-set samples — matches legacy train.py lines 344-354
# (iterates train_loader only, not all 1000 samples)
pod_train_list = []
for inputs, _ in train_loader:
    pod_train_list.append(inputs)
pod_norm_torch = torch.cat(pod_train_list, dim=0)   # shape: (800, NUM_POD_COEFFS)

avg_err, max_err, timing = compare_gradients_nature_style_dataset(
    model, pod_norm_torch, device=DEVICE,
    h=1e-2, max_modes=20,
    save_path=os.path.join(RESULTS_DIR, 'gradient_comparison.pdf'),
    batch_size=16,   # matches legacy train.py line 368
)
print(f'Mean rel. error: {avg_err:.6f}, Max rel. error: {max_err:.6f}')
print(f'AD/FD speedup: {timing["speedup_ratio"]:.1f}x')


Dataset-level gradient comparison for 800 samples...
  Batch 1/50 (samples 0–15)


  Batch 2/50 (samples 16–31)
  Batch 3/50 (samples 32–47)
  Batch 4/50 (samples 48–63)
  Batch 5/50 (samples 64–79)
  Batch 6/50 (samples 80–95)
  Batch 7/50 (samples 96–111)
  Batch 8/50 (samples 112–127)
  Batch 9/50 (samples 128–143)
  Batch 10/50 (samples 144–159)
  Batch 11/50 (samples 160–175)
  Batch 12/50 (samples 176–191)
  Batch 13/50 (samples 192–207)
  Batch 14/50 (samples 208–223)
  Batch 15/50 (samples 224–239)
  Batch 16/50 (samples 240–255)
  Batch 17/50 (samples 256–271)
  Batch 18/50 (samples 272–287)
  Batch 19/50 (samples 288–303)
  Batch 20/50 (samples 304–319)
  Batch 21/50 (samples 320–335)
  Batch 22/50 (samples 336–351)
  Batch 23/50 (samples 352–367)
  Batch 24/50 (samples 368–383)
  Batch 25/50 (samples 384–399)
  Batch 26/50 (samples 400–415)
  Batch 27/50 (samples 416–431)
  Batch 28/50 (samples 432–447)
  Batch 29/50 (samples 448–463)
  Batch 30/50 (samples 464–479)
  Batch 31/50 (samples 480–495)
  Batch 32/50 (samples 496–511)
  Batch 33/50 (samples 512–

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/core/gradient_analysis.py:327: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(x="POD Mode", y="Gradient Difference (AD - FD)",


Mean rel. error: 0.117614, Max rel. error: 100.000000
AD/FD speedup: 88.8x


## 6 · Compute Gradients for All Samples

In [8]:
gradients = compute_all_gradients(model, pod_coeffs, DEVICE, batch_size=32)
print(f'Gradient matrix shape: {gradients.shape}')

grad_save = os.path.join(RESULTS_DIR, 'pod_gradients.npy')
np.save(grad_save, gradients)
print(f'Saved to {grad_save}')

Normalised POD range: [-1.0000, 1.0000]
  Gradient batch 1/32 (samples 0–31)
  Gradient batch 2/32 (samples 32–63)
  Gradient batch 3/32 (samples 64–95)
  Gradient batch 4/32 (samples 96–127)
  Gradient batch 5/32 (samples 128–159)
  Gradient batch 6/32 (samples 160–191)
  Gradient batch 7/32 (samples 192–223)
  Gradient batch 8/32 (samples 224–255)
  Gradient batch 9/32 (samples 256–287)
  Gradient batch 10/32 (samples 288–319)
  Gradient batch 11/32 (samples 320–351)
  Gradient batch 12/32 (samples 352–383)
  Gradient batch 13/32 (samples 384–415)
  Gradient batch 14/32 (samples 416–447)
  Gradient batch 15/32 (samples 448–479)
  Gradient batch 16/32 (samples 480–511)
  Gradient batch 17/32 (samples 512–543)
  Gradient batch 18/32 (samples 544–575)
  Gradient batch 19/32 (samples 576–607)
  Gradient batch 20/32 (samples 608–639)
  Gradient batch 21/32 (samples 640–671)
  Gradient batch 22/32 (samples 672–703)
  Gradient batch 23/32 (samples 704–735)
  Gradient batch 24/32 (samples 73

## 7 · Active Subspace Analysis

In [9]:
# All-sample min/max for gradient scaling — matches legacy main.py lines 27-46
XX_as_min = np.min(pod_coeffs, axis=0)
XX_as_max = np.max(pod_coeffs, axis=0)
scale = (XX_as_max - XX_as_min) / 2.0
scale[scale < 1e-10] = 1.0
gradients_scaled = gradients * scale

# Bootstrap-based AS computation — nboot=1000 matches legacy main.py
ss = ac.subspaces.Subspaces()
ss.compute(df=gradients_scaled, nboot=1000)

# Plot first 6 eigenvalues, subspace errors, and eigenvectors — matches legacy main.py exactly
opts = ac.utils.plotters.plot_opts(savefigs=True)
ac.utils.plotters.eigenvalues(
    ss.eigenvals[:6],
    e_br=ss.e_br[:6, :],
    out_label='$C_d$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvalues.jpg'),
)
ac.utils.plotters.subspace_errors(
    ss.sub_br[:6, :], out_label='$C_d$', opts=opts
)
ac.utils.plotters.eigenvectors(
    ss.eigenvecs[:6, :2],
    out_label='$C_d$',
    opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'eigenvectors.jpg'),
)

# Partition: use the 2-D active subspace
n_active = 2
ss.partition(n_active)
print(f'Active subspace dimension: {n_active}')
print(f'W1 (active directions) shape: {ss.W1.shape}')

Active subspace dimension: 2
W1 (active directions) shape: (150, 2)


## 8 · Sufficient Summary Plot and Mode Importance

In [10]:
# Load drag coefficient (de-normalised)
qoi_raw = np.loadtxt(QOI_DATA_PATH)
qoi_full = qoi_raw[:pod_coeffs.shape[0], 1]

# Project onto active subspace (XX_as_min/max from above — all-sample min/max)
pod_norm_all = 2.0 * (pod_coeffs - XX_as_min) / (XX_as_max - XX_as_min) - 1.0
y_active = pod_norm_all @ ss.W1

ac.utils.plotters.sufficient_summary(
    y_active, qoi_full.reshape(-1, 1),
    out_label='$C_d$', opts=opts,
    save_path=os.path.join(RESULTS_DIR, 'sufficient_summary.jpg'),
)

# POD mode importance — weighted sum then normalise
pod_importance = np.sum(ss.eigenvecs[:, :n_active] ** 2 * ss.eigenvals[:n_active, 0], axis=1)
total = pod_importance.sum()
if total > 0:
    pod_importance = pod_importance / total
else:
    pod_importance = np.ones(NUM_POD_COEFFS) / NUM_POD_COEFFS

plot_pod_importance(
    NUM_POD_COEFFS, pod_importance,
    save_dir=os.path.join(RESULTS_DIR, 'Importance'),
    top_n=6,
)

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/utils/visualization.py:178: UserWarning: First parameter to grid() is false, but line properties are supplied. The grid will be enabled.
  ax.grid(False, axis='x', alpha=0.3)


'../results/Case1_Cylinder/Importance/pod_mode_importance_150_top6.jpg'

## 9 · Subspace–Polynomial R² Heatmap

In [11]:
# Select the 2 most important POD modes
n_dim_range = 2
n_poly_range = 3
n_importance = np.argsort(pod_importance)[-n_dim_range:][::-1]

XX_as_23 = pod_norm_all[:, n_importance]          # (N, 2)
eigenvecs_23 = ss.eigenvecs[n_importance, :]      # (2, k)

heatmap_path, heatmap_data = plot_subspace_polynomial_heatmap(
    XX_as_23,
    qoi_full.reshape(-1, 1),
    eigenvecs_23,
    n_dim_range=n_dim_range,
    n_poly_range=n_poly_range,
    save_dir=os.path.join(RESULTS_DIR, 'Heatmap'),
)

r2_matrix = heatmap_data['r2_matrix']
best_idx = np.unravel_index(np.nanargmax(r2_matrix), r2_matrix.shape)
best_dim, best_poly = best_idx[0] + 1, best_idx[1] + 1
best_r2 = r2_matrix[best_idx]
print(f'Heatmap saved to {heatmap_path}')
print(f'Best R²={best_r2:.6f}  →  subspace dim={best_dim}, poly order={best_poly}')
print('R² matrix:\n', r2_matrix)

/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

Heatmap saved to ../results/Case1_Cylinder/Heatmap/subspace_polynomial_heatmap.jpg
Best R²=0.999565  →  subspace dim=2, poly order=3
R² matrix:
 [[0.96757567 0.96749593 0.96744796]
 [0.96787791 0.995171   0.99956524]]


## 10 · Activity Scores and Interaction Heatmap

In [12]:
# Activity scores for the first 6 modes
top_n = 6
alpha_D = (ss.eigenvals[:n_active].reshape(1, n_active) * ss.eigenvecs[:, :n_active] ** 2).sum(axis=1)
alpha_D_top = alpha_D[:top_n]
print(f'Activity scores (first {top_n}):', alpha_D_top)
print('Normalised:                     ', alpha_D_top / alpha_D_top.sum())

# Lower-triangle modal interaction heatmap
heatmap_fig = plot_interaction_heatmap(
    ss.eigenvecs,
    ss.eigenvals,
    pod_importance,
    n_active=n_active,
    top_n=top_n,
    save_dir=os.path.join(RESULTS_DIR, 'Activity_Score'),
)
print(f'Interaction heatmap saved to {heatmap_fig}')

Activity scores (first 6): [0.00758123 0.15652881 1.24035422 0.03182617 0.00194789 0.01240794]
Normalised:                      [0.00522611 0.10790281 0.85503562 0.0219393  0.00134277 0.00855339]
Interaction heatmap saved to ../results/Case1_Cylinder/Activity_Score/param_interaction_heatmap_lower_top6.jpg


## 11 · Polynomial Response Surface

In [ ]:
from lib.active_subspaces.utils.rs import PolynomialApproximation
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Project onto best_dim-dimensional subspace using only the top-importance modes
# (matches legacy main.py line 177: y_23 = XX_as[:, n_importance].dot(W[n_importance, :best_dim]))
y_23 = pod_norm_all[:, n_importance].dot(ss.eigenvecs[n_importance, :best_dim])

X_train, X_test, f_train, f_test = train_test_split(
    y_23, qoi_full.reshape(-1, 1), test_size=0.2, random_state=42
)

# Cross-validate polynomial order 1–8 (matches legacy main.py lines 193-210)
n_values, r2_values, rmse_values = [], [], []
best_cv_score, best_cv_rmse, best_cv_n = -1, float('inf'), 1

for n in range(1, 4):
    rs_cv = PolynomialApproximation(N=n)
    rs_cv.train(X_train, f_train)
    pred  = rs_cv.predict(X_test)[0]
    score = r2_score(f_test, pred)
    rmse  = np.sqrt(mean_squared_error(f_test, pred))
    print(f'N={n}: R²={score:.6f}, RMSE={rmse:.8f}')
    n_values.append(n); r2_values.append(score); rmse_values.append(rmse)
    if score > best_cv_score or (abs(score - best_cv_score) < 1e-4 and rmse < best_cv_rmse):
        best_cv_score, best_cv_rmse, best_cv_n = score, rmse, n

print(f'\nBest poly order (CV): N={best_cv_n}, R²={best_cv_score:.6f}')

plot_polynomial_cv(
    n_values, r2_values, rmse_values, best_cv_n, best_cv_score, best_cv_rmse,
    save_dir=os.path.join(RESULTS_DIR, 'Polynomial_CV'),
)

# Train final RS with best_poly from the R² heatmap (matches legacy main.py line 219)
RS = PolynomialApproximation(N=best_poly)
RS.train(X_train, f_train)
print(f'Final RS trained with N={best_poly}, train R²={RS.Rsqr:.6f}')

# Build 2-D visualisation grid (only when best_dim >= 2)
if y_23.shape[1] >= 2:
    xx, yy = np.meshgrid(
        np.linspace(y_23[:, 0].min(), y_23[:, 0].max(), 50),
        np.linspace(y_23[:, 1].min(), y_23[:, 1].max(), 50),
    )
    n_dims     = y_23.shape[1]
    grid_full  = np.zeros((xx.size, n_dims))
    grid_full[:, :2] = np.column_stack([xx.ravel(), yy.ravel()])
    zz = RS.predict(grid_full)[0].reshape(xx.shape)

    plot_response_surface_2d(
        xx, yy, zz, y_23, qoi_full,
        results_dir=os.path.join(RESULTS_DIR, 'PRS'),
    )
else:
    print(f'y_23 is 1-D (best_dim={best_dim}); skipping 2-D surface plot.')

validate_response_surface(
    RS, X_test, f_test,
    save_dir=os.path.join(RESULTS_DIR, 'RS_Validation'),
)

compare_rom_fom_predictions(
    y_23, qoi_full, RS,
    t_start=250.0, dt=0.05,
    qoi_label='$C_d$',
    geometry='cylinder',
    scatter_xlim=(1.349, 1.37),
    save_dir=os.path.join(RESULTS_DIR, 'ROM_FOM'),
)

N=1: R²=0.967878, RMSE=0.00115764
N=2: R²=0.995171, RMSE=0.00044885
N=3: R²=0.999565, RMSE=0.00013468

Best poly order (CV): N=3, R²=0.999565


/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  poly_weights = np.linalg.lstsq(B, f)[0]
/mnt/data/bak/HD5/YDW/POD-AS-ROM/POD-AS-PRS-Github-2/lib/active_subspaces/utils/rs.py:105: FutureWarning: `rcond` parameter will change to the default of machine precisio

Final RS trained with N=3, train R²=0.999604

Test samples: 200
  R²   = 0.999565
  RMSE = 0.00013468
  MAE  = 0.00009633
Metrics saved to: ../results/Case1_Cylinder/RS_Validation/validation_metrics.txt
ROM vs FOM: R²=0.9996, RMSE=0.000135, MAE=0.007703
Performance metrics saved to: ../results/Case1_Cylinder/ROM_FOM/performance_metrics.txt


{'r2': 0.9995968413583102,
 'rmse': 0.0001350258040063188,
 'mae': 0.00770324620205807}